# 01 - Exploratory Data Analysis

Explore synthetic (or exported production) failed-payment data to understand:
- failure-reason distribution and retryability mix
- amount bands vs failure modes
- time-of-day patterns

Generate a dataset first if needed:
`python scripts/generate_test_data.py --count 2000 --out data/batch.json`

In [ ]:
import json
from pathlib import Path

import pandas as pd

path = Path('../data/batch.json')
if not path.exists():
    import sys
    sys.path.insert(0, '..')
    from app.utils.mock_data import generate_payment_batch
    batch = generate_payment_batch(2000, seed=7)
else:
    batch = json.loads(path.read_text())

rows = [{
    'amount': p['amount'],
    'method': p['method'],
    'status': p['status'],
    'reason_code': p.get('failure_reason_code'),
    'attempt_number': p['attempt_number'],
} for p in batch]
df = pd.DataFrame(rows)
df.head()

In [ ]:
failed = df[df['status'] == 'failed']
print(f"failure share: {len(failed)/len(df):.1%} of {len(df)} payments")
failed['reason_code'].value_counts(normalize=True).plot(kind='barh', title='Failure reason distribution');

In [ ]:
import numpy as np
failed['amount_band'] = pd.cut(failed['amount'], [0, 1000, 5000, 25000, 10**9], labels=['<1k','1k-5k','5k-25k','25k+'])
pd.crosstab(failed['amount_band'], failed['method'], normalize='index').round(3)

**Takeaways to validate against real data**
- `insufficient_funds` + `authentication_timeout` usually dominate -> delayed retries & digital nudges carry most recoverable value.
- High-value failures concentrate in card/netbanking -> IVR + CRM escalation pays off there.